# 🏭 Simulasi Monte Carlo: Estimasi Probabilitas Jumlah Produk Cacat Melebihi Batas Toleransi

**Mata Kuliah:** Pemodelan & Simulasi  
**Dataset:** Product Defects Dataset (Kaggle)  
**Pertanyaan Bisnis:** Jika dalam satu bulan produksi terdapat 1.000 unit produk, berapa probabilitas jumlah produk cacat melebihi batas toleransi 200 unit?

---

## 📋 Outline
1. Import Library
2. Load & Eksplorasi Data (EDA)
3. Data Cleaning
4. Ekstraksi Parameter (μ, σ)
5. Visualisasi Distribusi Data Asli
6. Membangun Simulasi Monte Carlo
7. Analisis Risiko & Kesimpulan

## 1. 📦 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print('✅ Library berhasil diimport!')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

## 2. 📊 Load & Eksplorasi Data (EDA)

In [ ]:
# Load dataset
df = pd.read_csv('defects_data.csv')
df['defect_date'] = pd.to_datetime(df['defect_date'])

print('=' * 60)
print('📂 INFORMASI DATASET')
print('=' * 60)
print(f'Jumlah baris   : {df.shape[0]:,}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'Kolom          : {df.columns.tolist()}')
print(f'Periode data   : {df["defect_date"].min().date()} s/d {df["defect_date"].max().date()}')
print()
df.head(10)

In [ ]:
# Statistik deskriptif
print('=' * 60)
print('📈 STATISTIK DESKRIPTIF')
print('=' * 60)
df.describe()

In [ ]:
# Distribusi kategorikal
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribusi Variabel Kategorikal', fontsize=16, fontweight='bold', y=1.02)

# Defect Type
defect_counts = df['defect_type'].value_counts()
axes[0].pie(defect_counts.values, labels=defect_counts.index, autopct='%1.1f%%',
            colors=['#FF6B6B', '#4ECDC4', '#45B7D1'], startangle=90)
axes[0].set_title('Tipe Cacat', fontweight='bold')

# Severity
sev_counts = df['severity'].value_counts()
colors_sev = {'Minor': '#2ecc71', 'Moderate': '#f39c12', 'Critical': '#e74c3c'}
axes[1].bar(sev_counts.index, sev_counts.values,
            color=[colors_sev[s] for s in sev_counts.index], edgecolor='white', linewidth=1.5)
axes[1].set_title('Tingkat Keparahan (Severity)', fontweight='bold')
axes[1].set_ylabel('Jumlah Cacat')
for i, (idx, val) in enumerate(zip(sev_counts.index, sev_counts.values)):
    axes[1].text(i, val + 5, str(val), ha='center', fontweight='bold')

# Inspection Method
insp_counts = df['inspection_method'].value_counts()
axes[2].barh(insp_counts.index, insp_counts.values,
             color=['#9B59B6', '#3498DB', '#1ABC9C'])
axes[2].set_title('Metode Inspeksi', fontweight='bold')
axes[2].set_xlabel('Jumlah')
for i, val in enumerate(insp_counts.values):
    axes[2].text(val + 2, i, str(val), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_eda_kategorikal.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot EDA kategorikal tersimpan!')

## 3. 🧹 Data Cleaning

In [ ]:
print('=' * 60)
print('🔍 CEK MISSING VALUES')
print('=' * 60)
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

# Cek outlier pada repair_cost menggunakan IQR
Q1 = df['repair_cost'].quantile(0.25)
Q3 = df['repair_cost'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['repair_cost'] < lower_bound) | (df['repair_cost'] > upper_bound)]
print(f'\n🔍 CEK OUTLIER (IQR Method) pada repair_cost:')
print(f'  Lower bound : {lower_bound:.2f}')
print(f'  Upper bound : {upper_bound:.2f}')
print(f'  Jumlah outlier: {len(outliers)}')

df_clean = df.copy()
print(f'\n✅ Dataset bersih! Total record: {len(df_clean):,}')

## 4. 📐 Ekstraksi Parameter (μ, σ)

**Variabel Acak yang Dipilih:** Jumlah produk cacat per bulan (monthly defect count)

Kami menghitung berapa banyak produk cacat yang terdeteksi setiap bulan dari data historis, kemudian menggunakan nilai rata-rata (μ) dan standar deviasi (σ) sebagai parameter distribusi untuk simulasi.

In [ ]:
# Hitung jumlah cacat per bulan sebagai variabel acak
df_clean['month'] = df_clean['defect_date'].dt.to_period('M')
monthly_defects = df_clean.groupby('month').size().reset_index(name='defect_count')
monthly_defects['month_str'] = monthly_defects['month'].astype(str)

print('=' * 60)
print('📊 JUMLAH CACAT PER BULAN (Data Historis)')
print('=' * 60)
print(monthly_defects[['month_str', 'defect_count']].to_string(index=False))

# Ekstrak parameter
mu = monthly_defects['defect_count'].mean()
sigma = monthly_defects['defect_count'].std()
minimum = monthly_defects['defect_count'].min()
maximum = monthly_defects['defect_count'].max()

print()
print('=' * 60)
print('🎯 PARAMETER EKSTRAKSI')
print('=' * 60)
print(f'  Mean (μ)              : {mu:.2f} produk cacat/bulan')
print(f'  Std Dev (σ)           : {sigma:.2f} produk cacat/bulan')
print(f'  Minimum               : {minimum} produk cacat')
print(f'  Maksimum              : {maximum} produk cacat')
print(f'  Coefficient of Var.   : {(sigma/mu)*100:.1f}%')

BATCH_SIZE = 1000
TOLERANCE = 200
defect_rate_hist = mu / BATCH_SIZE

print()
print('=' * 60)
print('⚙️  PARAMETER BISNIS')
print('=' * 60)
print(f'  Ukuran batch produksi : {BATCH_SIZE:,} unit/bulan')
print(f'  Batas toleransi       : {TOLERANCE} unit ({TOLERANCE/BATCH_SIZE*100:.0f}%)')
print(f'  Defect rate historis  : {defect_rate_hist:.2%}')

## 5. 📊 Visualisasi Distribusi Data Asli

In [ ]:
# Uji normalitas dengan Shapiro-Wilk
stat_sw, p_sw = stats.shapiro(monthly_defects['defect_count'])
print(f'Shapiro-Wilk Test: statistic={stat_sw:.4f}, p-value={p_sw:.4f}')
if p_sw > 0.05:
    print('→ Data berdistribusi Normal (p > 0.05) ✅')
else:
    print('→ Data tidak normal (p ≤ 0.05), namun karena n kecil, CLT mendukung penggunaan Normal')

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Analisis Distribusi Data Historis – Jumlah Cacat per Bulan',
             fontsize=14, fontweight='bold')

x_range = np.linspace(minimum - 20, maximum + 20, 300)
normal_pdf = stats.norm.pdf(x_range, mu, sigma)

# Bar chart bulanan
axes[0].bar(monthly_defects['month_str'], monthly_defects['defect_count'],
            color='#3498DB', edgecolor='white', linewidth=1.5, alpha=0.85)
axes[0].axhline(mu, color='#E74C3C', linestyle='--', linewidth=2, label=f'Mean (μ={mu:.0f})')
axes[0].axhline(TOLERANCE, color='#F39C12', linestyle=':', linewidth=2.5, label=f'Batas Toleransi ({TOLERANCE})')
axes[0].set_title('Jumlah Cacat per Bulan (Historis)', fontweight='bold')
axes[0].set_xlabel('Bulan')
axes[0].set_ylabel('Jumlah Produk Cacat')
axes[0].legend(fontsize=9)
axes[0].tick_params(axis='x', rotation=30)
for i, val in enumerate(monthly_defects['defect_count']):
    axes[0].text(i, val + 1.5, str(val), ha='center', fontsize=9, fontweight='bold')

# Histogram
axes[1].hist(monthly_defects['defect_count'], bins=6, density=True,
             color='#2ECC71', edgecolor='white', alpha=0.8, label='Data Historis')
axes[1].plot(x_range, normal_pdf, 'r-', linewidth=2.5, label=f'Normal(μ={mu:.0f}, σ={sigma:.1f})')
axes[1].axvline(TOLERANCE, color='#F39C12', linestyle=':', linewidth=2.5, label=f'Batas={TOLERANCE}')
axes[1].set_title('Histogram + Distribusi Normal', fontweight='bold')
axes[1].set_xlabel('Jumlah Cacat')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)

# Boxplot repair_cost per severity
severity_order = ['Minor', 'Moderate', 'Critical']
colors_box = ['#2ECC71', '#F39C12', '#E74C3C']
bp = axes[2].boxplot(
    [df_clean[df_clean['severity'] == s]['repair_cost'] for s in severity_order],
    labels=severity_order, patch_artist=True,
    boxprops=dict(linewidth=1.5),
    medianprops=dict(color='white', linewidth=2)
)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[2].set_title('Distribusi Biaya Perbaikan per Severity', fontweight='bold')
axes[2].set_xlabel('Tingkat Keparahan')
axes[2].set_ylabel('Biaya Perbaikan (Rp)')

plt.tight_layout()
plt.savefig('plot_distribusi_asli.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot distribusi data asli tersimpan!')

## 6. 🎲 Membangun Simulasi Monte Carlo

**Model:** Jumlah produk cacat per bulan disimulasikan menggunakan Distribusi Normal dengan parameter dari data historis:
- Distribusi: **Normal**
- Parameter: **μ = 166.67**, **σ = 14.60**
- Iterasi: **10.000 skenario**

In [ ]:
# ============================================================
#  SIMULASI MONTE CARLO
# ============================================================
N_SIMULATIONS = 10_000
np.random.seed(42)

print('=' * 60)
print('🎲 KONFIGURASI SIMULASI MONTE CARLO')
print('=' * 60)
print(f'  Distribusi         : Normal')
print(f'  Mean (μ)           : {mu:.4f}')
print(f'  Std Dev (σ)        : {sigma:.4f}')
print(f'  Jumlah iterasi     : {N_SIMULATIONS:,}')
print(f'  Batas toleransi    : {TOLERANCE} produk cacat')
print()

# Pembangkitan angka acak
simulated_defects = np.random.normal(loc=mu, scale=sigma, size=N_SIMULATIONS)
simulated_defects = np.clip(simulated_defects, 0, None)

print(f'✅ Simulasi selesai! {N_SIMULATIONS:,} skenario berhasil dibangkitkan.')
print()
print('📊 Statistik Hasil Simulasi:')
print(f'  Mean hasil sim.    : {simulated_defects.mean():.2f}')
print(f'  Std Dev hasil sim. : {simulated_defects.std():.2f}')
print(f'  Minimum            : {simulated_defects.min():.2f}')
print(f'  Maksimum           : {simulated_defects.max():.2f}')
print(f'  Persentil 5%       : {np.percentile(simulated_defects, 5):.2f}')
print(f'  Persentil 95%      : {np.percentile(simulated_defects, 95):.2f}')

## 7. 📉 Analisis Risiko & Kesimpulan

In [ ]:
# ============================================================
#  HITUNG PROBABILITAS
# ============================================================
exceed_count = np.sum(simulated_defects > TOLERANCE)
prob_exceed = exceed_count / N_SIMULATIONS
prob_safe = 1 - prob_exceed
z_score = (TOLERANCE - mu) / sigma
prob_exceed_theoretical = 1 - stats.norm.cdf(z_score)

print('=' * 60)
print('🎯 HASIL ANALISIS RISIKO')
print('=' * 60)
print(f'  Batas toleransi               : {TOLERANCE} produk cacat')
print(f'  Z-score                       : {z_score:.4f}')
print()
print(f'  ❗ P(cacat > {TOLERANCE}) simulasi  : {prob_exceed:.4f} = {prob_exceed*100:.2f}%')
print(f'  ❗ P(cacat > {TOLERANCE}) teoritis  : {prob_exceed_theoretical:.4f} = {prob_exceed_theoretical*100:.2f}%')
print(f'  ✅ P(cacat ≤ {TOLERANCE}) aman      : {prob_safe:.4f} = {prob_safe*100:.2f}%')
print()

if prob_exceed < 0.05:
    level = '🟢 RENDAH'
elif prob_exceed < 0.20:
    level = '🟡 SEDANG'
elif prob_exceed < 0.50:
    level = '🟠 TINGGI'
else:
    level = '🔴 SANGAT TINGGI'

print('=' * 60)
print('💡 INTERPRETASI RISIKO')
print('=' * 60)
print(f'  Level Risiko  : {level}')
print(f'  Dari {N_SIMULATIONS:,} skenario, sebanyak {exceed_count:,} ({prob_exceed*100:.2f}%)')
print(f'  menghasilkan cacat yang MELEBIHI batas toleransi {TOLERANCE} unit.')
print()
print(f'  📌 Agar risiko < 5%: batas toleransi minimal {np.percentile(simulated_defects, 95):.0f} unit')
print(f'  📌 Agar risiko < 1%: batas toleransi minimal {np.percentile(simulated_defects, 99):.0f} unit')

In [ ]:
# ============================================================
#  VISUALISASI HASIL SIMULASI
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hasil Simulasi Monte Carlo – Risiko Produk Cacat',
             fontsize=15, fontweight='bold')

# Plot 1: Histogram
ax1 = axes[0]
n, bins, patches = ax1.hist(simulated_defects, bins=60, color='#3498DB',
                             edgecolor='white', linewidth=0.5, alpha=0.8,
                             density=True, label='Hasil Simulasi')
for patch, left_edge in zip(patches, bins[:-1]):
    if left_edge >= TOLERANCE:
        patch.set_facecolor('#E74C3C')
        patch.set_alpha(0.9)

ax1.axvline(TOLERANCE, color='#E74C3C', linestyle='--', linewidth=2.5,
            label=f'Batas Toleransi = {TOLERANCE}')
ax1.axvline(mu, color='#2ECC71', linestyle='-', linewidth=2,
            label=f'Mean = {mu:.1f}')

x_sim = np.linspace(simulated_defects.min(), simulated_defects.max(), 300)
ax1.plot(x_sim, stats.norm.pdf(x_sim, mu, sigma), 'k-', linewidth=2, alpha=0.7,
         label='Distribusi Normal Teoritis')

x_risk = np.linspace(TOLERANCE, simulated_defects.max(), 200)
ax1.fill_between(x_risk, 0, stats.norm.pdf(x_risk, mu, sigma),
                 color='#E74C3C', alpha=0.3, label=f'Area Risiko ({prob_exceed*100:.2f}%)')

ax1.set_title(f'Distribusi Jumlah Cacat dari {N_SIMULATIONS:,} Simulasi', fontweight='bold')
ax1.set_xlabel('Jumlah Produk Cacat per Bulan')
ax1.set_ylabel('Density')
ax1.legend(fontsize=9)
ax1.text(TOLERANCE + 1, ax1.get_ylim()[1] * 0.85,
         f'P(cacat > {TOLERANCE}) = {prob_exceed*100:.2f}%',
         color='#E74C3C', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Plot 2: CDF
ax2 = axes[1]
sorted_sim = np.sort(simulated_defects)
cdf = np.arange(1, N_SIMULATIONS + 1) / N_SIMULATIONS
ax2.plot(sorted_sim, cdf * 100, color='#3498DB', linewidth=2.5, label='CDF Simulasi')
ax2.plot(x_sim, stats.norm.cdf(x_sim, mu, sigma) * 100, 'k--', linewidth=2,
         alpha=0.7, label='CDF Teoritis')
ax2.axvline(TOLERANCE, color='#E74C3C', linestyle='--', linewidth=2.5,
            label=f'Batas = {TOLERANCE}')
ax2.axhline(prob_safe * 100, color='#E74C3C', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.scatter([TOLERANCE], [prob_safe * 100], color='#E74C3C', zorder=5, s=80)
ax2.annotate(f'  ({TOLERANCE}, {prob_safe*100:.1f}%)',
             xy=(TOLERANCE, prob_safe * 100),
             color='#E74C3C', fontsize=10, fontweight='bold')
ax2.set_title('Kurva CDF – Probabilitas Kumulatif', fontweight='bold')
ax2.set_xlabel('Jumlah Produk Cacat per Bulan')
ax2.set_ylabel('Probabilitas Kumulatif (%)')
ax2.legend(fontsize=9)
ax2.set_ylim(0, 100)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
plt.savefig('plot_monte_carlo_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot hasil simulasi tersimpan!')

In [ ]:
# ============================================================
#  ANALISIS SENSITIVITAS THRESHOLD
# ============================================================
thresholds = np.arange(130, 220, 5)
probabilities = [np.mean(simulated_defects > t) * 100 for t in thresholds]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, probabilities, 'o-', color='#E74C3C', linewidth=2.5,
        markersize=7, markerfacecolor='white', markeredgewidth=2)
ax.axvline(TOLERANCE, color='#F39C12', linestyle='--', linewidth=2,
           label=f'Batas Toleransi = {TOLERANCE}')
ax.axhline(5, color='#2ECC71', linestyle=':', linewidth=2,
           label='Threshold risiko rendah (5%)')
ax.fill_between(thresholds, probabilities, alpha=0.15, color='#E74C3C')
ax.set_title('Analisis Sensitivitas: Probabilitas Melebihi Batas vs Nilai Batas Toleransi',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Batas Toleransi Cacat (unit)', fontsize=11)
ax.set_ylabel('Probabilitas Melebihi Batas (%)', fontsize=11)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('plot_sensitivitas.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot sensitivitas tersimpan!')

In [ ]:
# ============================================================
#  RINGKASAN EKSEKUTIF
# ============================================================
print('=' * 65)
print('               📋 RINGKASAN EKSEKUTIF')
print('=' * 65)
print('STUDI KASUS: Simulasi Monte Carlo – Risiko Produk Cacat')
print('DATASET   : Product Defects Dataset – 1.000 catatan cacat')
print('PERIODE   : Januari 2024 – Juni 2024 (6 bulan)')
print()
print('VARIABEL ACAK: Jumlah produk cacat per bulan')
print(f'  → Distribusi: NORMAL (μ={mu:.2f}, σ={sigma:.2f})')
print()
print(f'SIMULASI  : {N_SIMULATIONS:,} skenario | Batch: {BATCH_SIZE:,} unit | Toleransi: {TOLERANCE} unit')
print()
print('HASIL UTAMA:')
print(f'  ⚠️  P(cacat > {TOLERANCE}) : {prob_exceed*100:.2f}%')
print(f'  ✅  P(cacat ≤ {TOLERANCE}) : {prob_safe*100:.2f}%')
print(f'  📊  Z-score          : {z_score:.2f}')
print(f'  🏷️  Level Risiko     : {level}')
print()
print('REKOMENDASI:')
print(f'  • Alert jika cacat > {np.percentile(simulated_defects, 90):.0f} unit/bulan (P90)')
print(f'  • Investigasi penuh jika cacat > {TOLERANCE} unit/bulan')
print('=' * 65)